# Programmation Python  pour les mathématiques

**Julien Guillod**, [Sorbonne Université](http://www.sorbonne-universite.fr/),
Licence <a href="https://creativecommons.org/licenses/by-nc-nd/4.0/">CC BY-NC-ND</a>

L'entier des chapitres est disponible au format
[HTML](https://python.guillod.org//) et [PDF](https://python.guillod.org//python.pdf).
Ce notebook peut également être exécuté sur [GESIS](https://notebooks.gesis.org/binder/v2/gh/guillod/python/master?filepath=chap06.ipynb).

# 6 Algèbre

<div id="ch:algebre"></div>

Tout d'abord une méthode de résolution de système linéaire par un algorithme direct est étudié, puis ensuite une méthode itérative sera utilisée pour déterminer le vecteur propre associé à la plus grande valeur propre d'une matrice. Finalement, les groupes générés par un ensemble de permutations seront étudiés.

**Concepts abordés:**

* méthode de résolution directe (décomposition LU)

* algorithme *in place*

* méthode itérative (puissance itérée)

* groupes de permutations

* orbite et stabilisateur



In [1]:
import numpy as np
import scipy.linalg
A = np.array([[2,-1,0],[-1,2,-1],[0,-1,2]]) # un exemple de wikipedia: https://fr.wikipedia.org/wiki/D%C3%A9composition_LU
B = np.array([[2,4,3,5],[-4,-7,-5,-8],[6,8,2,9],[4,9,-2,14]]) # un exemple de https://www.youtube.com/watch?v=BFYFkn-eOQk

# Exercice 6.1: Décomposition LU

<div id="exer:algebre-LU"></div>

La décomposition LU consiste à décomposer une matrice $A$ de taille $n \times n$ sous la forme $A=LU$ où $L$ est une matrice triangulaire inférieure avec des 1 sur la diagonale et $U$ une matrice triangulaire supérieure. Une fois la décomposition $A=LU$ d'une matrice connue, il est alors très facile de résoudre le problème linéaire $A\boldsymbol{x} = \boldsymbol{b}$ en résolvant d'abord $L\boldsymbol{y} = \boldsymbol{b}$ puis $U\boldsymbol{x} = \boldsymbol{y}$. L'avantage de la décomposition LU sur l'algorithme de Gauss par exemple, est qu'une fois la décomposition LU connue, il est possible de résoudre le système linéaire rapidement avec des seconds membres différents.

Vu que $l_{ik}=0$ si $k>i$, nous avons:

$$
a_{ij} = \sum_{k=1}^n l_{ik}u_{kj} = l_{ii}u_{ij} + \sum_{k=1}^{i-1} l_{ik}u_{kj} \,,
$$

et donc comme $l_{ii}=1$:

$$
u_{ij} = a_{ij} - \sum_{k=1}^{i-1} l_{ik}u_{kj} \,.
$$

D'un autre côté, vu que $u_{kj}=0$ si $k>j$, alors:

$$
a_{ij} = \sum_{k=1}^n l_{ik}u_{kj} = l_{ij}u_{jj} + \sum_{k=1}^{j-1} l_{ik}u_{kj} \,,
$$

et donc si $u_{jj}\neq0$

$$
l_{ij} = \frac{1}{u_{jj}} \left( a_{ij} - \sum_{k=1}^{j-1} l_{ik}u_{kj} \right) \,.
$$

Ainsi, si les $(i-1)$ premières lignes de $U$ et les $(i-1)$ premières colonnes de $L$ sont connues, il est possible de déterminer la $i$-ème ligne de $U$ par:

$$
u_{ij} = a_{ij} - \sum_{k=1}^{i-1} l_{ik}u_{kj} \,, \quad j \geq i \,,
$$

puis la $i$-ème colonne de $L$ par:

$$
l_{ji} = \frac{1}{u_{ii}} \left( a_{ji} - \sum_{k=1}^{i-1} l_{jk}u_{ki} \right) \,, \quad j > i\  \,.
$$

Pour que la décomposition LU d'une matrice $A$ soit possible il faut que les $u_{ii}$ ne soient jamais nul. Cela est effectivement le cas lorsque la matrice $A$ et toutes ses sous-matrices principales sont inversibles.

**a)**
Écrire une fonction `LU(A)` qui retourne le résultat de la décomposition LU d'une matrice.



En transcrivant les formules pour les les coefficients de L et de U: $l_{ij} = \frac{1}{u_{jj}} \left( a_{ij} - \sum_{k=1}^{j-1} l_{ik}u_{kj} \right)$ et $u_{ij} = a_{ij} - \sum_{k=1}^{i-1} l_{ik}u_{kj} \,, \quad j \geq i \,,$
Test avec l'exemple de wikipedia: https://fr.wikipedia.org/wiki/D%C3%A9composition_LU
$$A=\begin{pmatrix} 2  &   -1  &   0  \\ -1  &    2  &   1 \\ 0  &   -1  &    2 \\ \end{pmatrix}$$

In [2]:
def LU(A):
    """
    array -> array x array
    Hypothèse: matrice A carrée doit être inversible et toutes ses sous-matrices principales
    Retourne la décomposition LU de A
    """
    n = A.shape[0]
    L = np.identity(n, dtype = float)
    U = np.zeros((n,n), dtype = float)
    for j in range(n): # colonne
        for i in range(j+1): # j >= i
            somme = sum(L[i][k] * U[k][j] for k in range(i)) # fait la somme pour le calcul du coefficent de U
            U[i][j] = A[i][j] - somme
            # print("i={0}, j={1}, somme={2}, U[i][j]={3}".format(i,j,somme,U[i][j]))
        for i in range(j+1, n): # j > i
            print((i,j))
            somme = sum(L[i][k] * U[k][j]  for k in range(j)) # fait la somme pour le calcul du coefficient de L
            L[i][j] = (A[i][j] - somme) / U[j][j]
            # print("i={0}, j={1}, somme={2}, L[i][j]={3}".format(i,j,somme,L[i][j]))
    return L,U
    
A = np.array([[2,-1,0],[-1,2,-1],[0,-1,2]])
L,U=LU(A)
display(L,U)
print('----------------------------------------------------')
B = np.array([[2,4,3,5],[-4,-7,-5,-8],[6,8,2,9],[4,9,-2,14]])
L,U=LU(B)
display(L,U)

(1, 0)
(2, 0)
(2, 1)


array([[ 1.        ,  0.        ,  0.        ],
       [-0.5       ,  1.        ,  0.        ],
       [ 0.        , -0.66666667,  1.        ]])

array([[ 2.        , -1.        ,  0.        ],
       [ 0.        ,  1.5       , -1.        ],
       [ 0.        ,  0.        ,  1.33333333]])

----------------------------------------------------
(1, 0)
(2, 0)
(3, 0)
(2, 1)
(3, 1)
(3, 2)


array([[ 1.,  0.,  0.,  0.],
       [-2.,  1.,  0.,  0.],
       [ 3., -4.,  1.,  0.],
       [ 2.,  1.,  3.,  1.]])

array([[ 2.,  4.,  3.,  5.],
       [ 0.,  1.,  1.,  2.],
       [ 0.,  0., -3.,  2.],
       [ 0.,  0.,  0., -4.]])


**b)**
Donné la décomposition LU d'une matrice $A$, écrire une fonction `solve(L,U,b)` qui résout le système linéaire $A\boldsymbol{x} =  \boldsymbol{b}$.



Pour la matrice de l'exemple $A= \begin{pmatrix} 2  &   -1  &   0  \\ -1  &    2  &   1 \\ 0  &   -1  &    2 \\ \end{pmatrix}$ et la décomposition LU: $L= \begin{pmatrix} 1  &    0  &    0 \\ -1/2  &    1  &    0 \\ 0  &   -2/3  &    1 \\ \end{pmatrix}$ et $U= \begin{pmatrix} 2  &   -1   &   0  \\ 0  &   3/2  &   -1 \\ 0  &   0    &  4/3 \\ \end{pmatrix}$

In [ ]:
def solve(L,U,b):
    """ array x array x vector --> vector
        Hypothèse: L et U carrées de taille n et b vecteur de R^n
        Retourne la solution du système linéaire comme vecteur de R^n
    """
    n = L.shape[0]
    y = np.zeros(n, dtype = float)
    x = np.zeros(n, dtype = float)
    # fait Ly = b
    # y[0]=b[0]
    # y[1]=b[1]-L[1][0]*y[0]
    # y[2]=b[2]-(L[2][0]*y[0]+L[2][1]*y[1])
    # ou y = np.linalg.solve(L,b)
    for i in range(n):
        somme = sum(L[i][j] * y[j] for j in range(i))
        y[i] = b[i] - somme
    # print(y)
    
    # fait Ux = y
    # x[2]=y[2]/U[2][2]
    # x[1]=(y[1]-U[1][2]*x[2])/U[1][1]
    # x[0]=(y[0]-(U[0][1]*x[1]+U[0][2]*x[2])/U[0][0])
    # ou directement, x = np.linalg.solve(U,y)
    for i in reversed(range(n)):
        somme = sum(U[i][j] * x[j] for j in range(i+1, n))
        x[i] = (y[i] - somme) / U[i][i]
    
    return x

L = np.array([[ 1.        ,  0.        ,  0.        ],[-0.5       ,  1.        ,  0.        ],[ 0.        , -0.66666667,  1.        ]])
U = np.array([[ 2.        , -1.        ,  0.        ],[ 0.        ,  1.5       , -1.        ],[ 0.        ,  0.        ,  1.33333333]])
b = np.array([2,1,1])
print("valeur de x:",solve(L,U,b))


valeur de x: [2.25       2.50000001 1.75000001]



**c)**
Écrire une fonction `LU_inplace(A)` qui ne crée par des nouveaux tableaux pour $L$ et $U$ mais modifie $A$ de sorte que sa partie triangulaire inférieure (sans la diagonale) corresponde à $L$ et sa partie triangulaire supérieure (avec la diagonale) corresponde à $U$. Faire également une version `solve_inplace` prenant en entrée la sortie de `LU_inplace` et retournant la solution $\boldsymbol{x}$, sans utiliser de nouveaux tableaux.




**d)**
En utilisant la décomposition LU de la matrice $A$, écrire une fonction `det(A)` qui retourne son déterminant.



In [ ]:
def det(A):
    n = A.shape[0]
    L,U = LU(A)
    return (prod(U[i][i] for i in range(n))) # produit des coefficients de la diagonale de U = det(A)
    
print(np.linalg.det(A) == det(A))

True


# Exercice 6.2: Méthode de la puissance itérée

<div id="exer:algebre-power"></div>

Le but de cet exercice est de déterminer le vecteur propre d'une matrice associé à la valeur propre de plus grand module, en supposant que celle-ci est unique. Etant donné une matrice réelle $A$ de taille $n\times n$ et un vecteur $\boldsymbol{x}_0\in\mathbb{R}^n$, la suite de vecteurs $(\boldsymbol{x}_k)_ {k\in\mathbb{N}}$ est définie par:

$$
\boldsymbol{x}_{k+1} = \frac{A\boldsymbol{x}_k}{\Vert A\boldsymbol{x}_k\Vert} \,.
$$

En supposant par exemple que la matrice $A$ soit diagonalisable, alors il est possible de montrer que la suite $(\boldsymbol{x}_k)_ {k\in\mathbb{N}}$ converge à un signe près vers le vecteur propre associé à la plus grande valeur propre de $A$ en valeur absolue.

**Indication:**
La première étape est de remarquer que

$$
\boldsymbol{x}_k = \frac{A^2 \boldsymbol{x}_{k-2}}{\Vert A^2 \boldsymbol{x}_{k-2}\Vert} = \dots = \frac{A^k \boldsymbol{x}_0}{\Vert A^k \boldsymbol{x}_0\Vert} \,.
$$

Vu que $A$ est diagonalisable, soit ($\boldsymbol{v}_1, \boldsymbol{v}_2, \dots, \boldsymbol{v}_n)$ une base de vecteurs propres de $A$ associés aux valeurs propres $\lambda_1, \lambda_2, \dots, \lambda_n$. Sans perte de généralité, on suppose que $\lambda_1$ est la valeur propre de plus grand module, c'est-à-dire $|\lambda_1|>\max(|\lambda_2|,|\lambda_3|,\dots,|\lambda_n|)$. A noter que cela implique que $\lambda_1$ est réelle. Le vecteur $\boldsymbol{x}_0$ se décompose dans la base:

$$
\boldsymbol{x}_0 = \sum_{i=1}^n c_i \boldsymbol{v}_i \,,
$$

ainsi en supposant que $c_1\neq0$:

$$
A^k \boldsymbol{x}_0 = \sum_{i=1}^n c_i \lambda_i^k \boldsymbol{v}_i
= c_1 \lambda_1^k \left(\boldsymbol{v}_1 + \sum_{i=2}^n \frac{c_i}{c_1} \left(\frac{\lambda_i}{\lambda_1}\right)^k \boldsymbol{v}_i \right) .
$$

Vu que $|\lambda_1| > |\lambda_i|$ pour $i\geq2$, alors:

$$
\lim_{k\to\infty}\left(\boldsymbol{v}_1 + \sum_{i=2}^n \frac{c_i}{c_1} \left(\frac{\lambda_i}{\lambda_1}\right)^k \boldsymbol{v}_i \right) = \boldsymbol{v}_1 \,,
$$

puisque $\left|\frac{\lambda_i}{\lambda_1}\right| < 1$.
Par conséquent,

$$
\lim_{k\to\infty}\operatorname{sign}{(\lambda_1)}^k \boldsymbol{x}_k = \lim_{k\to\infty} \left(\frac{\lambda_1}{|\lambda_1|}\right)^k \frac{A^k \boldsymbol{x}_0}{\Vert A^k \boldsymbol{x}_0\Vert} = \operatorname{sign}{c_1} \frac{\boldsymbol{v}_1}{\Vert\boldsymbol{v}_1\Vert} \,.
$$

En choisissant $\boldsymbol{x}_0$ aléatoirement, alors avec probabilité un $c_1\neq0$ et donc la suite $(\boldsymbol{x}_k)_ {k\in\mathbb{N}}$ converge à un signe près vers un vecteur propre normalisé associé à la valeur propre de plus grand module.

**a)**
Définir une fonction `puissance(A, x0, k)` qui retourne $\boldsymbol{x}_k$. Avec cette fonction, déterminer le plus grand vecteur propre de la matrice

$$
A = \begin{pmatrix}0.5 & 0.5\\ 
0.2 & 0.8
\end{pmatrix} \,.
$$


**Réponse:**
Le plus grand vecteur propre est $\pm(0.70710678, 0.70710678)$



In [ ]:
def puissance(A, x0, k):
    for _ in range(k):
        nv = np.dot(A, x0)
        x0 = nv / np.linalg.norm(nv)
        # print(np.dot(np.dot(A,x0), x0))
    return list(np.around(x0,8))

# test avec la matrice A avec 20 iterations
assert puissance(np.array([[0.5, 0.5], [0.2, 0.8]]), np.random.random_sample((2,)), 20)==[0.70710678, 0.70710678]


**b)**
Déterminer la valeur propre associée au vecteur propre précédent.

**Indication:**
Si $\boldsymbol{v}$ est un vecteur propre normalisé de $A$, alors la valeur propre associée est donnée par $\lambda = A\boldsymbol{v}\cdot\boldsymbol{v}$.



In [ ]:
vecteurpropre = np.array(puissance(np.array([[0.5, 0.5], [0.2, 0.8]]), np.random.random_sample((2,)), 10))
valpropre = np.dot(np.dot(np.array([[0.5, 0.5], [0.2, 0.8]]),vecteurpropre), vecteurpropre)
print(valpropre)

1.0000010700358695



**c)**
Écrire un programme permettant de calculer automatiquement la valeur propre de plus grand module et le vecteur propre associé d'une matrice carrée avec une certaine précision donnée.



In [ ]:
def puissance1(A, x0, k, precision):
    if A.shape[0] != A.shape[1]: # la matrice doit être carrée
        return None
    iteration = 0
    val = 0
    ancval = 1
    while  not (iteration > k or abs(val - ancval) < precision):
        ancval = val
        nv = np.dot(A, x0)
        x0 = nv / np.linalg.norm(nv)
        val = np.dot(np.dot(A ,x0), x0)
        print(iteration, abs(val - ancval))
        iteration = iteration + 1
    return x0, val
    
V,val = puissance1(np.array([[0, 6, -1], [6, 0, 6],[-1, 6, 0]]), np.random.random_sample((3,)), 200, 1E-10)
print('%0.3f\t' %(val))
print(V)

0 2.036974887595329
1 0.9406701410991605
2 0.9830490286949423
3 0.9998215722236423
4 0.9892161333546978
5 0.9523719685824843
6 0.8931090187587123
7 0.8171214057201239
8 0.7308940741974332
9 0.6406683743932957
10 0.5516943221236197
11 0.4678570858492188
12 0.3916349793871676
13 0.3242785446410039
14 0.2660930528314296
15 0.21673470457643162
16 0.17546824849349463
17 0.14136514237898723
18 0.11344158354641287
19 0.09074576670079715
20 0.0724068292337865
21 0.05765726157962803
22 0.0458383172239607
23 0.03639543001607315
24 0.028868429306720955
25 0.02287962579916858
26 0.018121612422410394
27 0.01434579701044214
28 0.011352154531245162
29 0.008980365829996728
30 0.007102327390002827
31 0.0056159221615299515
32 0.004439900704936051
33 0.003509712197331538
34 0.002774131988839912
35 0.002192547637488218
36 0.0017327835750879927
37 0.001369362904295457
38 0.001082121836319061
39 0.0008551073079061666
40 0.0006757011963944848
41 0.0005339253607452576
42 0.0004218906744917206
43 0.00033336052


**d)**
Regarder la documentation de Numpy pour trouver les fonctions permettant de calculer les vecteurs propres et valeurs propres d'une matrice.



In [ ]:
 # https://numpy.org/doc/stable/reference/generated/numpy.linalg.eig.html

 print(np.linalg.eig((np.array([[3, -2], [3, 1]])))) # retourne les valuers et vecteurs propres
 print(np.linalg.eig((np.array([[0, 6, -1], [6, 0, 6],[-1, 6, 0]])))) # retourne les valuers et vecteurs propres

(array([2.+2.23606798j, 2.-2.23606798j]), array([[0.25819889+0.57735027j, 0.25819889-0.57735027j],
       [0.77459667+0.j        , 0.77459667-0.j        ]]))
(array([-9.,  1.,  8.]), array([[ 5.14495755e-01, -7.07106781e-01,  4.85071250e-01],
       [-6.85994341e-01,  2.03919322e-16,  7.27606875e-01],
       [ 5.14495755e-01,  7.07106781e-01,  4.85071250e-01]]))



**e)**
Comparer la rapidité du code précédent et des fonctions Numpy pour des matrices de tailles $n\times n$ avec $n=10,100,1000$.

**Indication:**
En prenant par exemple des matrices dont les coefficients sont générés aléatoirement dans l'intervalle $(0,1)$, le théorème de Perron-Frobenius assure l'existence d'une unique valeur propre de module maximal, et celle-ci est positive.



In [ ]:
%%timeit
A = np.random.random_sample((10,10))
puissance1(A, np.random.random_sample((10,)), 200, 1E-10)

136 µs ± 1.27 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [ ]:
%%timeit
A = np.random.random_sample((10,10))
np.linalg.eig(A)

54.8 µs ± 1.85 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [ ]:
%%timeit
A = np.random.random_sample((100,100))
puissance1(A, np.random.random_sample((100,)), 1000, 1E-10)

1.28 ms ± 2.92 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [ ]:
%%timeit
A = np.random.random_sample((100,100))
np.linalg.eig(A)

11 ms ± 70.2 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
%%timeit
A = np.random.random_sample((1000,1000))
puissance1(A, np.random.random_sample((1000,)), 1000, 1E-10)

12.2 ms ± 81.5 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
%%timeit
A = np.random.random_sample((1000,1000))
np.linalg.eig(A)

1.46 s ± 18.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Je pense que ce n'est pas vraiment representatif car j'ai pris la fonction Numpy eig() qui calcule tous les valeurs propres et vecteurs propres associés.

# Exercice 6.3: Groupes de permutations

<div id="exer:algebre-permutations"></div>

Le but est d'étudier les groupes de permutations en développant des algorithmes pour les caractériser. Un groupe de permutation $G \subset S_n$ est défini comme étant généré par un certain nombre de permutations: $G = \langle g_1, g_2,\dots,g_k \rangle$, avec $g_i\in S_n$ une permutation de l'ensemble $\{1,2,\dots,n\}$. Une permutation

$$
g = \begin{pmatrix}1 & 2 & 3 & 4\\ 
4 & 1 & 2 & 3
\end{pmatrix} \,,
$$

peut être représentée en python par le tuple `g = (0, 4, 1, 2, 3)`. Le zéro est ajouté afin de se conformer avec l'indexation à partir de zéro de Python et ainsi de simplifier un peu les implémentations. Cela veut simplement dire que le sommet `0` va sur le sommet `0`. A noter que cet exercice se prête particulièrement bien à une implémentation orientée objet, et dans ce cas les questions peuvent être adaptées en conséquence.

**a)**
Écrire une fonction `product(g1,g2)` qui calcule le produit de deux permutations.




**b)**
Écrire une fonction `inverse(g)` qui calcule l'inverse d'une permutation.




**c)**
Écrire une fonction qui retourne la décomposition d'une permutation sous forme de cycles représentés par une liste du tuples.




**d)**
Écrire une fonction qui retourne la permutation correspondant à une liste de cycles, c'est-à-dire qui fait l'inverse de la fonction précédente.




**e)**
<span style="color:red">!</span> En python un groupe $G = \langle g_1,g_2,\dots,g_k \rangle$ engendré par une famille de permutations, peut être représenté par une liste de permutations `G = [g1,g2,...,gk]`. Écrire une fonction `orbit(G,x)` qui retourne l'orbite d'un point $x\in\{1,2,\dots,n\}$:

$$
O_x = Gx = \{gx, g \in G\} \,.
$$



**Indication:**
Ne pas calculer l'ensemble des éléments du groupe, cela fait une liste beaucoup trop grande. Construire la liste $(X^0,X^1,\dots,X^N)$ d'ensembles disjoints définie récursivement par $X^0 = \{x\}$ et $X^n$ comme l'ensemble des éléments nouveaux de la forme $g_i y$ avec $1 \leq i \leq k$ et $y\in X^{n-1}$:

$$
X^{n}=\left(\bigcup_{i=1}^{k}g_{i}X^{n-1}\right)\setminus\left(\bigcup_{i=1}^{n-1}X^{i}\right) \,.
$$


**f)**
<span style="color:red">!!</span> Définir une fonction `stabilizer(G,x)` qui retourne le stabilisateur d'un point $x\in\{1,2,\dots,n\}$:

$$
G_x = \{g \in G : g x = x \} \,,
$$

sous la forme d'un ensemble de générateurs.

**Indication:**
Utiliser le théorème ou lemme de Schreier.

